# **Практическая работа №5. Работа с растровыми данными в Rasterio**

---

**Данные для обработки:**

- Одноканальное изображение (ЦМР): https://github.com/opengeos/datasets/releases/download/raster/dem_90m.tif
- Мультиспектральное изображение (Landsat): https://github.com/opengeos/datasets/releases/download/raster/cog.tif

### **Задание 1. Чтение и исследование растровых данных**



1. Откройте одноканальное изображение ЦМР с использованием `rasterio`.


In [ ]:
from urllib.request import urlretrieve

import rasterio

base_url = "https://github.com/opengeos/datasets/releases/download/raster/"
dem_path = "dem_90m.tif"
urlretrieve(base_url + dem_path, dem_path)

with rasterio.open(dem_path) as src:
    dem_profile = src.profile
    dem_res = src.res
    dem_bounds = src.bounds
    dem_dtypes = src.dtypes


2. Извлеките и выведите метаданные растра, включая СК, разрешение, границы, количество каналов и типы данных.


In [ ]:
print(f"Система координат: {dem_profile['crs']}")
print(f"Разрешение: {dem_res[0]:.3f} × {dem_res[1]:.3f} м")
print(f"Границы (запад, юг, восток, север): {tuple(dem_bounds)}")
print(f"Количество каналов: {dem_profile['count']}")
print(f"Типы данных каналов: {', '.join(dem_dtypes)}")


3. Отобразите ширину и высоту растра, а также типы данных пикселей, чтобы понять размеры сетки и структуру данных.

In [ ]:
print(f"Ширина: {dem_profile['width']} пикселей")
print(f"Высота: {dem_profile['height']} пикселей")
print(f"Тип данных пикселей: {dem_profile['dtype']}")


---



### **Задание 2. Визуализация и манипулирование растровыми каналами**



1. Визуализируйте одноканальное изображение ЦМР с использованием пользовательской цветовой карты (например, cmap='terrain').


In [ ]:
import matplotlib.pyplot as plt
from rasterio.plot import plotting_extent

with rasterio.open(dem_path) as src:
    dem = src.read(1, masked=True)
    dem_extent = plotting_extent(src)

plt.figure(figsize=(8, 5))
plt.imshow(dem, cmap="terrain")
plt.title("Цифровая модель рельефа")
plt.xlabel("Столбец")
plt.ylabel("Строка")
plt.colorbar(label="Высота, м")
plt.tight_layout()
plt.show()


2. Откройте мультиспектральное изображение и визуализируйте первый канал с использованием подходящей цветовой карты.


In [ ]:
image_path = "cog.tif"
urlretrieve(base_url + image_path, image_path)

with rasterio.open(image_path) as src:
    bands = src.read()

first_band = bands[0]

plt.figure(figsize=(7, 4))
plt.imshow(first_band, cmap="gray", vmin=first_band.min(), vmax=first_band.max())
plt.title("Первый канал")
plt.xlabel("Столбец")
plt.ylabel("Строка")
plt.colorbar(label="Значения пикселей")
plt.tight_layout()
plt.show()


3. Объедините несколько каналов из мультиспектрального изображения (например, Красный, Зеленый и Синий) и совместите их в один массив для создания RGB-композитного изображения.

>*Примечание:*
>
> При работе с многоканальными растровыми изображениями, особенно в формате COG.TIFF (Cloud Optimized GeoTIFF), может возникнуть проблема с некорректным отображением псевдоцветного изображения. Это связано с тем, что разные каналы могут иметь различные диапазоны значений пикселей.
>
> Для корректной визуализации на шаге 2 необходимо применить нормализацию к каждому каналу перед отображением.



Включите следующую функцию в ваш код:

```python
# Функция для нормализации канала в диапазон от 0 до 1
def normalize(band):
    band_min = band.min()
    band_max = band.max()
    band_norm = (band - band_min) / (band_max - band_min)
    return band_norm
```

In [ ]:
import numpy as np


def normalize(band):
    band_min = band.min()
    band_max = band.max()
    band_norm = (band - band_min) / (band_max - band_min)
    return band_norm


red = normalize(bands[2])
green = normalize(bands[1])
blue = normalize(first_band)
rgb = np.dstack((red, green, blue))

plt.figure(figsize=(7, 4))
plt.imshow(rgb)
plt.title("RGB-композит (каналы 3, 2, 1)")
plt.xlabel("Столбец")
plt.ylabel("Строка")
plt.tight_layout()
plt.show()


---



### **Задание 3. Обрезка растра с использованием индексации массива**



1. Откройте мультиспектральное изображение и обрежьте его с использованием спискового среза (указав диапазоны строк и столбцов).


In [ ]:
from rasterio.windows import Window

row_start, row_end = 35, 170
col_start, col_end = 70, 260

with rasterio.open(image_path) as src:
    image = src.read()
    clipped = image[:, row_start:row_end, col_start:col_end]
    window = Window.from_slices((row_start, row_end), (col_start, col_end))
    clipped_transform = src.window_transform(window)
    clipped_profile = src.profile

print(f"Размер фрагмента: {clipped.shape[2]} × {clipped.shape[1]} пикселей")


2. Визуализируйте обрезанную часть изображения с использованием matplotlib, чтобы убедиться в корректном результате.


>*Примечание:*
>
> При работе с многоканальными растровыми изображениями, особенно в формате COG.TIFF (Cloud Optimized GeoTIFF), может возникнуть проблема с некорректным отображением псевдоцветного изображения. Это связано с тем, что разные каналы могут иметь различные диапазоны значений пикселей.
>
> Для корректной визуализации на шаге 2 необходимо применить нормализацию к каждому каналу перед отображением.



Включите следующую функцию в ваш код:

```python
# Функция для нормализации канала в диапазон от 0 до 1
def normalize(band):
    band_min = band.min()
    band_max = band.max()
    band_norm = (band - band_min) / (band_max - band_min)
    return band_norm
```

In [ ]:
red_clipped = normalize(clipped[2])
green_clipped = normalize(clipped[1])
blue_clipped = normalize(clipped[0])
rgb_clipped = np.dstack((red_clipped, green_clipped, blue_clipped))

plt.figure(figsize=(7, 4))
plt.imshow(rgb_clipped)
plt.title("Обрезанный фрагмент")
plt.axis("off")
plt.show()


3. Сохраните обрезанное подмножество растра в новый файл с именем `clipped_multispectral.tif`.

In [ ]:
clipped_profile.update(
    width=clipped.shape[2],
    height=clipped.shape[1],
    transform=clipped_transform,
)

clipped_path = "clipped_multispectral.tif"
with rasterio.open(clipped_path, "w", **clipped_profile) as dst:
    dst.write(clipped)

print(f"Сохранён файл {clipped_path}")


---



### **Задание 4. Вычисление NDWI (калькуляция каналов)**



1. Откройте мультиспектральное изображение и извлеките каналы Green (Зеленый) и Ближний инфракрасный (NIR).


In [ ]:
with rasterio.open(image_path) as src:
    green = src.read(2, masked=True, out_dtype="float32")
    nir = src.read(4, masked=True, out_dtype="float32")
    image_profile = src.profile
    image_extent = plotting_extent(src)


2. Вычислите Нормализованный разностный водный индекс ([NDWI](https://en.wikipedia.org/wiki/Normalized_difference_water_index)) по формуле:

    NDWI = (Green - NIR) / (Green + NIR)

In [ ]:
ndwi = np.ma.divide(green - nir, green + nir)

print(f"Диапазон NDWI: от {ndwi.min():.3f} до {ndwi.max():.3f}")


3. Визуализируйте результат NDWI с использованием цветовой карты, подходящей для воды (например, cmap='Blues'), чтобы выделить водные объекты.


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.imshow(ndwi, cmap="Blues", extent=image_extent, vmin=-1, vmax=1)
plt.title("Нормализованный разностный водный индекс")
plt.xlabel("Долгота, °")
plt.ylabel("Широта, °")
plt.ticklabel_format(useOffset=False)
plt.colorbar(label="NDWI")
plt.tight_layout()
plt.show()




4. Сохраните полученное изображение NDWI как новый растровый файл с именем ndwi.tif.

In [ ]:
ndwi_profile = image_profile.copy()
ndwi_profile.update(count=1, dtype="float32", nodata=np.nan)

ndwi_path = "ndwi.tif"
with rasterio.open(ndwi_path, "w", **ndwi_profile) as dst:
    dst.write(ndwi.filled(np.nan), 1)

print(f"Сохранён файл {ndwi_path}")


---



### **Задание 5. Перепроецирование растровых данных**



1. Перепроецируйте одноканальный растр ЦМР из его исходной СК в EPSG:4326 (WGS 84) с использованием функции `rasterio.warp.reproject`.


In [ ]:
from rasterio.warp import Resampling, calculate_default_transform, reproject

dst_crs = "EPSG:4326"

with rasterio.open(dem_path) as src:
    transform, width, height = calculate_default_transform(
        src.crs, dst_crs, src.width, src.height, *src.bounds
    )
    dem_reprojected = np.empty((height, width), dtype="float32")

    reproject(
        source=rasterio.band(src, 1),
        destination=dem_reprojected,
        src_transform=src.transform,
        src_crs=src.crs,
        src_nodata=src.nodata,
        dst_transform=transform,
        dst_crs=dst_crs,
        dst_nodata=np.nan,
        resampling=Resampling.bilinear,
    )


2. Сохраните перепроецированный растр в новый файл GeoTIFF с именем `reprojected_dem.tif`.


In [ ]:
reprojected_profile = dem_profile.copy()
reprojected_profile.update(
    crs=dst_crs, transform=transform, width=width, height=height,
    dtype="float32", nodata=np.nan,
)

reprojected_path = "reprojected_dem.tif"
with rasterio.open(reprojected_path, "w", **reprojected_profile) as dst:
    dst.write(dem_reprojected, 1)

print(f"Сохранён файл {reprojected_path}")


3. Визуализируйте как исходный, так и перепроецированный наборы данных ЦМР, чтобы сравнить, как перепроецирование влияет на пространственное покрытие и разрешение.

In [ ]:
with rasterio.open(reprojected_path) as src:
    reprojected_extent = plotting_extent(src)
    reprojected_res = src.res

print(f"Исходная ЦМР: {dem.shape[1]} × {dem.shape[0]} пикселей")
print(f"Разрешение: {dem_res[0]:.3f} × {dem_res[1]:.3f} м")
print(f"Перепроецированная ЦМР: {width} × {height} пикселей")
print(f"Разрешение: {reprojected_res[0]:.6f} × {reprojected_res[1]:.6f}°")

fig, axes = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")
vmin, vmax = dem.min(), dem.max()

axes[0].imshow(dem, cmap="terrain", extent=dem_extent, vmin=vmin, vmax=vmax)
axes[0].set_title(f"Исходная ЦМР ({dem_profile['crs']})")
axes[0].set_xlabel("X, м")
axes[0].set_ylabel("Y, м")
axes[0].ticklabel_format(style="plain", useOffset=False)
axes[0].locator_params(axis="x", nbins=5)

dem_plot = axes[1].imshow(
    dem_reprojected, cmap="terrain", extent=reprojected_extent, vmin=vmin, vmax=vmax
)
axes[1].set_title(f"Перепроецированная ЦМР ({dst_crs})")
axes[1].set_xlabel("Долгота, °")
axes[1].set_ylabel("Широта, °")
axes[1].ticklabel_format(style="plain", useOffset=False)
fig.colorbar(dem_plot, ax=axes, shrink=0.8, label="Высота, м")
plt.show()


---